# Task 12: Calibrate results against the published pattern

This notebook checks whether our fixed 663-occupation sample reproduces what two published comparisons report. The written calibration, with citations, is `data/datasets/processed/agreement_calibration.md`.

| Source | What it reports | Re-run here as |
| --- | --- | --- |
| The Budget Lab at Yale (Gimbel, Kendall & Kulsakdinun, 19 Feb 2026) | Ranks of Eloundou GPT-4 beta (`dv_rating_beta`) correlate at about 0.7–0.9 with each of six other modern metrics; R² of 0.47–0.78. Variance of z-scored metrics rises with a PCA-weighted exposure score (slope 0.0617). Frey–Osborne is not included. | **Check A** (rank correlations against `dv_rating_beta`) and **Check B** (variance of z-scores against exposure) |
| ILO research brief (Merola et al., 2026), Table 2, from del Rio-Chanona et al. (2025) | Correlation of each exposure measure with wages: Frey–Osborne −0.5576, Felten et al. (2018) 0.5375, Eloundou et al. (2024) 0.4243. | **Check C** (correlation of each index with BLS median annual wage) |

Section 4 traces individual occupations back to their source files, and section 5 tests how much one crosswalk choice matters.

Run top to bottom from the repository root or `notebooks/`. Results are unweighted and occupation-level. Exposure is not a prediction of job loss.

In [1]:
from pathlib import Path
import hashlib
import numpy as np
import pandas as pd
import scipy
from scipy import stats

project_folder = Path('..') if Path('../data/datasets/processed').is_dir() else Path('.')
data_folder = project_folder / 'data/datasets'
processed_folder = data_folder / 'processed'
sample_path = processed_folder / 'agreement_analysis_sample.csv'
source_sha256 = hashlib.sha256(sample_path.read_bytes()).hexdigest()
sample = pd.read_csv(sample_path, dtype={'soc2018': 'string'})

indices = ['aioe', 'fo_prob', 'dv_rating_beta', 'human_rating_beta']
assert len(sample) == 663 and sample['soc2018'].is_unique
assert sample['included_in_analysis'].eq(True).all()
assert sample[indices].notna().all().all()

pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 60)
print(f'Fixed sample verified: {len(sample)} occupations. SHA-256: {source_sha256}')
print(f'pandas {pd.__version__}; NumPy {np.__version__}; SciPy {scipy.__version__}')

Fixed sample verified: 663 occupations. SHA-256: 7f0c606895d080d913ed147c8436a60ea61cafeebd1dfe3a9bba907509f782ee
pandas 2.3.3; NumPy 2.5.4; SciPy 1.18.1


## 1. Check A: correlation level (Budget Lab)

The Budget Lab "compares the ranks of DV Rating Beta against each of the other metrics" and reports correlations of .7–.9 and R² between .47 and .78. A univariate regression on ranks has R² = ρ², so Spearman ρ and ρ² are directly comparable. Of their six comparison metrics, we have two: AIOE and `human_rating_beta`. Frey–Osborne is shown for contrast only; the Budget Lab did not use it.

In [2]:
check_a = []
for other, in_budget_lab in [('aioe', True), ('human_rating_beta', True), ('fo_prob', False)]:
    rho, p = stats.spearmanr(sample['dv_rating_beta'], sample[other])
    check_a.append({
        'x': 'dv_rating_beta', 'y': other, 'n': len(sample), 'spearman_rho': rho,
        'r_squared': rho ** 2, 'p_value': p, 'in_budget_lab': in_budget_lab,
        'inside_0.7_0.9': 0.7 <= rho <= 0.9,
    })
check_a = pd.DataFrame(check_a)
check_a.round(4)

,x,y,n,spearman_rho,r_squared,p_value,in_budget_lab,inside_0.7_0.9
0,dv_rating_beta,aioe,663,0.8628,0.7444,0.0,True,True
1,dv_rating_beta,human_rating_beta,663,0.9118,0.8313,0.0,True,False
2,dv_rating_beta,fo_prob,663,-0.2670,0.0713,0.0,False,False


## 2. Check B: does disagreement rise with exposure? (Budget Lab method)

Task 11 measured disagreement with *ranks*. The Budget Lab uses z-scores instead, explicitly because ranks "flatten the difference between occupations". Here we copy their approach:

1. z-score each metric across the 663 occupations.
2. **Specification 1:** regress the per-occupation variance of the z-scores (sample variance, ddof = 1) on their mean. The Budget Lab uses PCA weights; for AIOE + both Eloundou ratings, the first-component weights are almost equal (shown below), so an equal-weight mean is used throughout.
3. **Specification 2:** regress the variance of the *other* metrics on one Eloundou rating's z-score. This avoids putting the same metric on both sides.

Each is run for the modern set (AIOE + Eloundou), separately per label source, and again with Frey–Osborne added. The Budget Lab's slope (0.0617) comes from six metrics and controls for salary, share of women, and major occupation group, so only the sign and significance are compared, not the exact size.

In [3]:
z = sample[indices].apply(lambda v: (v - v.mean()) / v.std(ddof=0))

modern = ['aioe', 'dv_rating_beta', 'human_rating_beta']
eigenvalues, eigenvectors = np.linalg.eigh(np.corrcoef(z[modern].to_numpy().T))
first_pc = np.abs(eigenvectors[:, -1])
print('First-component weights (AIOE, GPT-4 beta, human beta):', (first_pc / first_pc.sum()).round(3))

def regress(x, y):
    fit = stats.linregress(x, y)
    return {'slope': fit.slope, 'p_value': fit.pvalue, 'spearman_rho': stats.spearmanr(x, y)[0]}

metric_sets = {
    'AIOE + GPT-4 beta': ['aioe', 'dv_rating_beta'],
    'AIOE + human beta': ['aioe', 'human_rating_beta'],
    'AIOE + GPT-4 beta + human beta': modern,
    'AIOE + GPT-4 beta + Frey–Osborne': ['aioe', 'dv_rating_beta', 'fo_prob'],
    'AIOE + human beta + Frey–Osborne': ['aioe', 'human_rating_beta', 'fo_prob'],
}
check_b = []
for name, columns in metric_sets.items():
    values = z[columns]
    check_b.append({'specification': '1: variance vs mean z', 'metrics': name,
                    'exposure_x': 'mean of the same z-scores', 'n': len(values),
                    **regress(values.mean(axis=1), values.var(axis=1, ddof=1))})
for x, others in [('dv_rating_beta', ['aioe', 'human_rating_beta']),
                  ('human_rating_beta', ['aioe', 'dv_rating_beta']),
                  ('dv_rating_beta', ['aioe', 'fo_prob']),
                  ('human_rating_beta', ['aioe', 'fo_prob'])]:
    check_b.append({'specification': '2: variance of others vs one metric', 'metrics': ' + '.join(others),
                    'exposure_x': x, 'n': len(z),
                    **regress(z[x], z[others].var(axis=1, ddof=1))})
check_b = pd.DataFrame(check_b)
check_b['positive_and_p_below_0.05'] = check_b['slope'].gt(0) & check_b['p_value'].lt(0.05)
check_b.round(4)

First-component weights (AIOE, GPT-4 beta, human beta): [0.331 0.333 0.336]


,specification,metrics,exposure_x,n,slope,p_value,spearman_rho,positive_and_p_below_0.05
0,1: variance vs mean z,AIOE + GPT-4 beta,mean of the same z-scores,663,0.0851,0.0000,0.2258,True
1,1: variance vs mean z,AIOE + human beta,mean of the same z-scores,663,0.0354,0.0001,0.1543,True
2,1: variance vs mean z,AIOE + GPT-4 beta + human beta,mean of the same z-scores,663,0.0663,0.0000,0.3419,True
3,1: variance vs mean z,AIOE + GPT-4 beta + Frey–Osborne,mean of the same z-scores,663,-0.0453,0.3620,-0.0558,False
4,1: variance vs mean z,AIOE + human beta + Frey–Osborne,mean of the same z-scores,663,-0.0120,0.8150,-0.0222,False
5,2: variance of others vs one metric,aioe + human_rating_beta,dv_rating_beta,663,0.0320,0.0002,0.1401,True
6,2: variance of others vs one metric,aioe + dv_rating_beta,human_rating_beta,663,0.0615,0.0000,0.1859,True
7,2: variance of others vs one metric,aioe + fo_prob,dv_rating_beta,663,-0.0070,0.8828,-0.0948,False
8,2: variance of others vs one metric,aioe + fo_prob,human_rating_beta,663,0.0919,0.0537,-0.0440,False


The rank-based Task 11 result is repeated here for comparison: median rank SD in the most-exposed band versus its shuffled-ranks baseline.

In [4]:
band_summary = pd.read_csv(processed_folder / 'exposure_band_summary.csv')
task11_top = band_summary.loc[band_summary['metric'].isin(['rank_sd', 'gap_aioe_elo'])
                              & band_summary['band'].isin([1, 5]),
    ['label_source', 'consensus_rule', 'metric', 'band', 'median', 'null_lo', 'null_hi', 'vs_null']]
task11_top.round(3)

,label_source,consensus_rule,metric,band,median,null_lo,null_hi,vs_null
0,gpt4,all_three,rank_sd,1,0.205,0.147,0.182,above
3,gpt4,all_three,gap_aioe_elo,1,0.080,0.175,0.264,below
20,gpt4,all_three,rank_sd,5,0.137,0.147,0.185,below
23,gpt4,all_three,gap_aioe_elo,5,0.109,0.173,0.265,below
25,gpt4,aioe_eloundou,rank_sd,1,0.259,0.166,0.217,above
28,gpt4,aioe_eloundou,gap_aioe_elo,1,0.075,0.154,0.222,below
45,gpt4,aioe_eloundou,rank_sd,5,0.264,0.167,0.212,above
48,gpt4,aioe_eloundou,gap_aioe_elo,5,0.109,0.150,0.219,below
50,human,all_three,rank_sd,1,0.205,0.148,0.183,above
53,human,all_three,gap_aioe_elo,1,0.070,0.176,0.263,below


## 3. Check C: correlation with wages (ILO Table 2)

The ILO brief reports how each measure correlates with wages. The statistic, wage source, and occupation set are not stated, so Pearson (raw and log wage) and Spearman are all shown. Wages are the BLS Employment Projections median annual wage for 2025 (Table 1.2, line items only), joined on exact 2018 SOC code. Six line items report "—" (no wage) and are left out.

ILO values: Frey–Osborne −0.5576, Felten et al. (2018) 0.5375, Eloundou et al. (2024) 0.4243. Our AIOE file is the Felten et al. (2021) appendix, which uses the same method as their 2018 paper.

In [5]:
bls = pd.read_excel(data_folder / 'bls_occupation.xlsx', sheet_name='Table 1.2', header=1)
bls = bls.loc[bls['Occupation type'].eq('Line item'),
              ['2025 National Employment Matrix code', 'Median annual wage, dollars, 2025']]
bls.columns = ['soc2018', 'median_wage_2025']
bls['median_wage_2025'] = pd.to_numeric(bls['median_wage_2025'], errors='coerce')
assert bls['soc2018'].is_unique
with_wages = sample.merge(bls, on='soc2018', how='left', validate='one_to_one')
with_wages = with_wages.dropna(subset=['median_wage_2025'])
print(f'Occupations with a 2025 median wage: {len(with_wages)} of {len(sample)}')

ilo_values = {'fo_prob': -0.5576, 'aioe': 0.5375, 'dv_rating_beta': 0.4243, 'human_rating_beta': 0.4243}
check_c = []
for index in indices:
    wage = with_wages['median_wage_2025']
    check_c.append({
        'index': index, 'n': len(with_wages),
        'pearson_wage': stats.pearsonr(with_wages[index], wage)[0],
        'pearson_log_wage': stats.pearsonr(with_wages[index], np.log(wage))[0],
        'spearman_wage': stats.spearmanr(with_wages[index], wage)[0],
        'ilo_reported': ilo_values[index],
    })
check_c = pd.DataFrame(check_c)
check_c['same_sign_as_ilo'] = np.sign(check_c[['pearson_wage', 'pearson_log_wage', 'spearman_wage']]).eq(
    np.sign(check_c['ilo_reported']), axis=0).all(axis=1)
check_c.round(4)

Occupations with a 2025 median wage: 656 of 663


,index,n,pearson_wage,pearson_log_wage,spearman_wage,ilo_reported,same_sign_as_ilo
0,aioe,656,0.4539,0.5174,0.5201,0.5375,True
1,fo_prob,656,-0.5150,-0.5593,-0.5394,-0.5576,True
2,dv_rating_beta,656,0.3208,0.3899,0.4335,0.4243,True
3,human_rating_beta,656,0.3337,0.4058,0.4343,0.4243,True


## 4. Trace occupations back to source values

Seven occupations, chosen to cover each kind of surprising result:

| Why chosen | Occupation |
| --- | --- |
| Most-exposed band, largest Frey–Osborne gap | 19-3039 Psychologists, All Other |
| Largest AIOE–Eloundou gap overall | 21-1013 Marriage and Family Therapists |
| Large AIOE–Eloundou gap, crosswalk changed | 51-9162 Computer Numerically Controlled Tool Programmers |
| Highest rank SD among crosswalk-changed occupations (a 2010 code split in two) | 19-3034 School Psychologists, with 19-3033 Clinical and Counseling Psychologists |
| Largest GPT-4 vs human difference | 31-9094 Medical Transcriptionists |
| 2010 code renumbered in 2018; human beta exactly 0 | 53-1041 Aircraft Cargo Handling Supervisors |

For each one, the raw value in each source file is matched to its 2010 contributor codes, the crosswalk handling, and the value in the analysis sample. The assertion confirms each sample value equals the unweighted mean of its raw contributors.

In [6]:
raw_aioe = pd.read_excel(data_folder / 'AIOE_appendixA.xlsx').rename(
    columns={'SOC Code': 'soc2010', 'AIOE': 'raw'}).set_index('soc2010')['raw']
raw_fo = pd.read_csv(data_folder / 'frey_osborne_probabilities.csv').rename(
    columns={'soc_code_2010': 'soc2010', 'fo_computerization_probability': 'raw'}).set_index('soc2010')['raw']
raw_eloundou = pd.read_csv(data_folder / 'eloundou_6digit.csv', dtype={'soc_6digit': 'string'}).set_index('soc_6digit')
crosswalked = {
    'aioe': pd.read_csv(processed_folder / 'aioe_2018soc.csv', dtype=str),
    'fo_prob': pd.read_csv(processed_folder / 'frey_osborne_2018soc.csv', dtype=str),
}
raw_sources = {'aioe': raw_aioe, 'fo_prob': raw_fo}
source_code_columns = {'aioe': 'aioe_source_codes', 'fo_prob': 'fo_source_codes'}
flag_columns = {'aioe': 'aioe_crosswalk_changed', 'fo_prob': 'fo_crosswalk_changed'}
rank_columns = {'aioe': 'rank_aioe', 'fo_prob': 'rank_fo',
                'dv_rating_beta': 'rank_elo_dv', 'human_rating_beta': 'rank_elo_hum'}

traced_socs = ['19-3039', '21-1013', '51-9162', '19-3034', '19-3033', '31-9094', '53-1041']
by_soc = sample.set_index('soc2018')
trace_rows = []
for soc in traced_socs:
    occupation = by_soc.loc[soc]
    for index in indices:
        row = {'soc2018': soc, 'title': occupation['title'], 'index': index,
               'sample_value': occupation[index], 'sample_rank': occupation[rank_columns[index]]}
        if index in raw_sources:
            codes = occupation[source_code_columns[index]].split(';')
            mapping = crosswalked[index].loc[crosswalked[index]['soc2018'].eq(soc)]
            raw_values = raw_sources[index].loc[codes]
            assert np.isclose(raw_values.mean(), occupation[index]), (soc, index)
            row.update({
                'source_2010_codes': ';'.join(codes),
                'raw_values': ';'.join(f'{v:.4g}' for v in raw_values),
                'crosswalk_handling': ';'.join(sorted(set(mapping['handling']))),
                'crosswalk_changed': occupation[flag_columns[index]],
            })
        else:
            assert np.isclose(raw_eloundou.loc[soc, index], occupation[index]), (soc, index)
            row.update({'source_2010_codes': '(Eloundou: published on 2018-based codes)',
                        'raw_values': f'{raw_eloundou.loc[soc, index]:.4g}',
                        'crosswalk_handling': 'none', 'crosswalk_changed': False})
        trace_rows.append(row)
traced = pd.DataFrame(trace_rows)
print('Every traced sample value equals the mean of its raw source values.')
traced.round(3)

Every traced sample value equals the mean of its raw source values.


,soc2018,title,index,sample_value,sample_rank,source_2010_codes,raw_values,crosswalk_handling,crosswalk_changed
0,19-3039,"Psychologists, All Other",aioe,1.390,0.959,19-3039,1.39,1:1,False
1,19-3039,"Psychologists, All Other",fo_prob,0.004,0.024,19-3039,0.0043,1:1,False
2,19-3039,"Psychologists, All Other",dv_rating_beta,0.454,0.700,(Eloundou: published on 2018-based codes),0.4544,none,False
3,19-3039,"Psychologists, All Other",human_rating_beta,0.282,0.537,(Eloundou: published on 2018-based codes),0.2817,none,False
4,21-1013,Marriage and Family Therapists,aioe,1.367,0.946,21-1013,1.367,1:1,False
5,21-1013,Marriage and Family Therapists,fo_prob,0.014,0.090,21-1013,0.014,1:1,False
6,21-1013,Marriage and Family Therapists,dv_rating_beta,0.111,0.259,(Eloundou: published on 2018-based codes),0.1111,none,False
7,21-1013,Marriage and Family Therapists,human_rating_beta,0.278,0.529,(Eloundou: published on 2018-based codes),0.2778,none,False
8,51-9162,Computer Numerically Controlled Tool Programmers,aioe,-0.658,0.354,51-4012;51-9199,-0.02418;-1.291,many:1;many:many,True
9,51-9162,Computer Numerically Controlled Tool Programmers,fo_prob,0.360,0.367,51-4012,0.36,many:1,True


## 5. Crosswalk sensitivity: "All Other" contributor codes

Tracing found that the Frey–Osborne source file has almost no 2010 "All Other" residual codes (ending in 9), while the AIOE file includes them. So when a 2018 occupation draws on a real parent code *plus* a catch-all code, AIOE averages in the catch-all score but Frey–Osborne does not. Here we list every affected occupation, recompute AIOE without its catch-all contributors (only where a specific contributor remains), and check whether any headline correlation moves.

In [7]:
catch_all = lambda code: code.endswith('99')
affected = []
aioe_without_catch_all = sample['aioe'].copy()
for i, row in sample.iterrows():
    codes = row['aioe_source_codes'].split(';')
    specific = [c for c in codes if not catch_all(c)]
    if len(codes) > 1 and any(map(catch_all, codes)) and specific:
        aioe_without_catch_all.loc[i] = raw_aioe.loc[specific].mean()
        affected.append({'soc2018': row['soc2018'], 'title': row['title'],
                         'aioe_source_codes': row['aioe_source_codes'],
                         'fo_source_codes': row['fo_source_codes'],
                         'aioe_sample': row['aioe'], 'aioe_without_catch_all': aioe_without_catch_all.loc[i]})
affected = pd.DataFrame(affected)
rank_before = sample['aioe'].rank(pct=True)
rank_after = aioe_without_catch_all.rank(pct=True)
affected['rank_sample'] = rank_before[sample['soc2018'].isin(affected['soc2018'])].to_numpy()
affected['rank_without_catch_all'] = rank_after[sample['soc2018'].isin(affected['soc2018'])].to_numpy()
print(f'{len(affected)} occupations average a catch-all code into AIOE.')
display(affected.round(3))

sensitivity = pd.DataFrame([{
    'pair': f'AIOE vs {other}',
    'spearman_sample': stats.spearmanr(sample['aioe'], sample[other])[0],
    'spearman_without_catch_all': stats.spearmanr(aioe_without_catch_all, sample[other])[0],
} for other in ['dv_rating_beta', 'human_rating_beta', 'fo_prob']])
sensitivity['change'] = sensitivity['spearman_without_catch_all'] - sensitivity['spearman_sample']
sensitivity.round(4)

10 occupations average a catch-all code into AIOE.


,soc2018,title,aioe_source_codes,fo_source_codes,aioe_sample,aioe_without_catch_all,rank_sample,rank_without_catch_all
0,13-2054,Financial Risk Specialists,13-2051;13-2099,13-2051;13-2099,1.371,1.381,0.949,0.952
1,15-1243,Database Architects,15-1141;15-1199,15-1141,1.259,1.285,0.873,0.889
2,15-1253,Software Quality Assurance Analysts and Testers,15-1132;15-1133;15-1199,15-1132;15-1133,1.239,1.242,0.864,0.867
3,15-1299,"Computer Occupations, All Other",15-1199;43-9011,43-9011,0.968,0.704,0.763,0.688
4,19-4044,Hydrologic Technicians,19-4041;19-4099,19-4041;19-4099,0.267,0.083,0.602,0.566
5,29-2099,"Health Technologists and Technicians, All Other",29-2054;29-2099,29-2054,-0.195,-0.106,0.496,0.522
6,29-9021,Health Information Technologists and Medical Registrars,29-2071;29-9099,29-2071,0.402,0.574,0.623,0.658
7,49-9099,"Installation, Maintenance, and Repair Workers, All Other",49-9093;49-9099,49-9093,-0.981,-1.373,0.213,0.086
8,51-9161,Computer Numerically Controlled Tool Operators,51-4011;51-9199,51-4011,-0.982,-0.674,0.211,0.348
9,51-9162,Computer Numerically Controlled Tool Programmers,51-4012;51-9199,51-4012,-0.658,-0.024,0.354,0.538


,pair,spearman_sample,spearman_without_catch_all,change
0,AIOE vs dv_rating_beta,0.8628,0.8644,0.0016
1,AIOE vs human_rating_beta,0.8669,0.8670,0.0000
2,AIOE vs fo_prob,-0.3678,-0.3690,-0.0012


## 6. Export

- `data/datasets/processed/calibration_results.csv`: Checks A, B and C in one tidy table.
- `data/datasets/processed/calibration_traced_occupations.csv`: section 4 trace.
- `data/datasets/processed/calibration_catch_all_codes.csv`: section 5 affected occupations.

The assertions confirm the patterns described in `agreement_calibration.md` still hold, so the write-up cannot silently go out of date.

In [8]:
results = pd.concat([
    check_a.assign(check='A: rank correlation with dv_rating_beta (Budget Lab)',
                   comparison=lambda d: d['x'] + ' vs ' + d['y'], value=lambda d: d['spearman_rho'],
                   published='0.7 to 0.9 (R² 0.47 to 0.78)'),
    check_b.assign(check='B: variance of z-scores vs exposure (Budget Lab)',
                   comparison=lambda d: 'spec ' + d['specification'].str[0] + ': ' + d['metrics']
                                        + ' | x = ' + d['exposure_x'],
                   value=lambda d: d['slope'], published='positive, significant (slope 0.0617)'),
    check_c.assign(check='C: correlation with median wage (ILO Table 2)',
                   comparison=lambda d: d['index'] + ' vs log wage (Pearson)',
                   value=lambda d: d['pearson_log_wage'], published=lambda d: d['ilo_reported'].astype(str)),
], ignore_index=True)[['check', 'comparison', 'n', 'value', 'p_value', 'published']]
results.to_csv(processed_folder / 'calibration_results.csv', index=False)
traced.to_csv(processed_folder / 'calibration_traced_occupations.csv', index=False)
affected.to_csv(processed_folder / 'calibration_catch_all_codes.csv', index=False)

# Patterns stated in agreement_calibration.md.
a = check_a.set_index('y')
assert a.loc['aioe', 'inside_0.7_0.9'] and a.loc['human_rating_beta', 'spearman_rho'] > 0.9
assert a.loc['fo_prob', 'spearman_rho'] < 0
assert check_b.loc[~check_b['metrics'].str.contains('Frey|fo_prob'), 'positive_and_p_below_0.05'].all()
assert not check_b.loc[check_b['specification'].str.startswith('1')
                       & check_b['metrics'].str.contains('Frey'), 'positive_and_p_below_0.05'].any()
assert check_c['same_sign_as_ilo'].all()
assert (check_c.set_index('index')['pearson_log_wage'] - pd.Series(ilo_values)).abs().max() < 0.05
assert sensitivity['change'].abs().max() < 0.01
for name in ['calibration_results.csv', 'calibration_traced_occupations.csv', 'calibration_catch_all_codes.csv']:
    print(f'Saved {processed_folder / name}')

Saved ../data/datasets/processed/calibration_results.csv
Saved ../data/datasets/processed/calibration_traced_occupations.csv
Saved ../data/datasets/processed/calibration_catch_all_codes.csv
